# NVIDIA Container Toolkit

A practical reference for running GPU-accelerated containers with the NVIDIA Container Toolkit — how it exposes host GPUs, drivers, and CUDA libraries to Docker, containerd, Podman, and Kubernetes workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

The **NVIDIA Container Toolkit** (package `nvidia-container-toolkit`, formerly `nvidia-docker2`) is the glue that lets unprivileged containers use the host's NVIDIA GPUs. It is the de-facto standard for GPU containers across the MLOps stack — every CUDA image on NGC, every PyTorch/TensorFlow training job in Kubernetes, and every Triton inference server ultimately relies on it.

### What is it?

It is a set of tools and a container-runtime *hook* that, at container start, injects the host's NVIDIA driver user-space libraries, CUDA driver (`libcuda.so`), device nodes (`/dev/nvidia*`), and management binaries (`nvidia-smi`) into the container's filesystem and namespaces. The container image itself ships only the CUDA *toolkit/runtime* (e.g. `libcudart`), never the driver — the driver always comes from the host. This decoupling is the whole point: one driver on the host serves containers built against many different CUDA toolkit versions.

### Why use it?

- **No driver in the image.** Images stay portable; you can run a CUDA 11.8 image and a CUDA 12.4 image on the same host with a single host driver (subject to driver ≥ toolkit compatibility).
- **Unprivileged GPU access.** No need for `--privileged` or hand-mounting `/dev/nvidia*` and dozens of `.so` files.
- **Standard runtime integration.** Works with Docker, containerd/`nerdctl`, Podman, and CRI-O, and underpins the Kubernetes device plugin and GPU Operator.
- **Fine-grained control.** Environment variables select which GPUs and which driver capabilities (compute, video, graphics, ...) a container sees.

### When to use it?

- Any time you run CUDA, cuDNN, NCCL, TensorRT, or Triton workloads in containers.
- Multi-tenant GPU nodes where you slice GPUs per container with `NVIDIA_VISIBLE_DEVICES`, MIG, or time-slicing.
- CI/CD that builds and tests GPU code in ephemeral containers.
- Kubernetes GPU scheduling (via the device plugin / GPU Operator, both of which sit on top of the toolkit).

## Key Features

### Core Capabilities of the NVIDIA Container Toolkit

| Feature | Description | Benefit |
|---------|-------------|---------|
| Driver/library injection | A runtime hook mounts host driver `.so` files, `libcuda.so`, and `/dev/nvidia*` into the container | Images never embed the driver; one host driver serves many CUDA versions |
| `--gpus` / device selection | Expose all or a subset of GPUs via the Docker `--gpus` flag or `NVIDIA_VISIBLE_DEVICES` | Per-container GPU isolation on shared nodes |
| Driver capabilities | `NVIDIA_DRIVER_CAPABILITIES` chooses compute, utility, graphics, video, display, etc. | Inject only the libraries a workload needs (e.g. NVENC for transcoding) |
| Multi-runtime support | Integrates with Docker, containerd, Podman, CRI-O via `nvidia-ctk runtime configure` | Same toolkit across the whole orchestration stack |
| Container Device Interface (CDI) | Generates a vendor-neutral CDI spec (`nvidia-ctk cdi generate`) consumed by Podman, containerd, CRI-O | Standards-based, runtime-agnostic GPU injection; no special runtime shim |
| MIG & device UUIDs | Select specific GPUs or MIG instances by index or UUID | Works with Multi-Instance GPU partitioning |
| `nvidia-ctk` CLI | One CLI to configure runtimes, generate CDI specs, and create symlinks/ldcache | Single entry point for setup and automation |

## Architecture Overview

### How a `docker run --gpus all` actually reaches the GPU

```
docker run --gpus all ...
        │
        ▼
Docker / containerd (high-level runtime)
        │   selects the "nvidia" OCI runtime
        ▼
nvidia-container-runtime  ──►  runc   (thin wrapper around runc)
        │   injects an OCI prestart hook into the container spec
        ▼
nvidia-container-runtime-hook   (a.k.a. nvidia-container-toolkit hook)
        │   reads NVIDIA_VISIBLE_DEVICES / capabilities from the env
        ▼
nvidia-container-cli  (CLI front-end of libnvidia-container)
        │
        ▼
libnvidia-container  ──►  mounts /dev/nvidia*, driver .so files,
                          libcuda.so, nvidia-smi into the container rootfs
                          and sets up cgroups for the selected devices
```

### Components

1. **`nvidia-container-runtime`** — a small shim that wraps the OCI runtime (`runc`). It edits the container's `config.json` to add the NVIDIA prestart hook, then hands off to `runc`. In *CDI mode* this shim is unnecessary because the high-level runtime applies CDI device edits directly.
2. **`nvidia-container-runtime-hook`** — the OCI prestart hook binary that the shim injects. It runs after the container's namespaces are created but before the user process starts, and calls `nvidia-container-cli`.
3. **`libnvidia-container` / `nvidia-container-cli`** — the library (and its CLI) that does the real work: discovers host driver components, bind-mounts the device nodes and libraries into the container, updates the dynamic linker cache, and configures device cgroups.
4. **`nvidia-ctk`** — the management CLI: `runtime configure` wires the toolkit into Docker/containerd/CRI-O, `cdi generate` produces CDI specs, and other subcommands handle symlinks and ldcache.
5. **Config file** — `/etc/nvidia-container-runtime/config.toml` controls the runtime mode (`legacy`, `cdi`, `csv`), the underlying `runc` path, debug logging, and accept-nvidia-visible-devices policy.

## Installation

### Prerequisites

- A supported **NVIDIA driver already installed on the host** (`nvidia-smi` works on the host). The toolkit injects the driver but does **not** install it.
- A container runtime: Docker Engine, containerd, Podman, or CRI-O.
- A Linux host (x86_64 or arm64/Jetson). The toolkit does not run on the Docker Desktop macOS/Windows VM, but works under WSL2 with the WSL CUDA driver.

### Installation Steps (Ubuntu/Debian, Docker)

The cell below is shell, not Python — run the commands on the host (or paste into a terminal). It adds the NVIDIA repo, installs the toolkit, configures the Docker runtime, and restarts Docker.

In [ ]:
# Host shell commands (Ubuntu/Debian + Docker). Run in a terminal, not the kernel.
INSTALL = r'''
# 1. Add the NVIDIA Container Toolkit apt repository (GPG key + source list)
curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey \
  | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg
curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list \
  | sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' \
  | sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list

# 2. Install the toolkit (pulls in libnvidia-container, the runtime, the hook, nvidia-ctk)
sudo apt-get update
sudo apt-get install -y nvidia-container-toolkit

# 3. Wire the "nvidia" runtime into the Docker daemon (edits /etc/docker/daemon.json)
sudo nvidia-ctk runtime configure --runtime=docker

# 4. Restart Docker so it picks up the new runtime
sudo systemctl restart docker

# 5. Smoke test: the container should print the host GPU table
sudo docker run --rm --gpus all nvidia/cuda:12.4.1-base-ubuntu22.04 nvidia-smi
'''
print(INSTALL)


## Basic Usage

### Quick Start

Once installed and configured, exposing GPUs is a single flag. Key knobs:

- `--gpus all` — expose every GPU. Equivalent to `NVIDIA_VISIBLE_DEVICES=all`.
- `--gpus '"device=0,1"'` — expose GPUs 0 and 1 (note the nested quoting in the Docker CLI).
- `--gpus 2` — expose any 2 GPUs.
- `-e NVIDIA_VISIBLE_DEVICES=GPU-<uuid>` — pin a specific physical GPU or MIG instance by UUID.
- `-e NVIDIA_DRIVER_CAPABILITIES=compute,utility` — choose which driver libraries get injected (default for CUDA base images).

The cell below renders the common command patterns as copy-pasteable shell.

In [ ]:
# Common `docker run` patterns for GPU containers.
patterns = {
    "All GPUs + nvidia-smi":
        "docker run --rm --gpus all nvidia/cuda:12.4.1-base-ubuntu22.04 nvidia-smi",
    "Specific GPUs (0 and 1)":
        "docker run --rm --gpus '\"device=0,1\"' nvidia/cuda:12.4.1-base-ubuntu22.04 nvidia-smi",
    "Pin a GPU by UUID":
        "docker run --rm -e NVIDIA_VISIBLE_DEVICES=GPU-4a1b... nvidia/cuda:12.4.1-base-ubuntu22.04 nvidia-smi -L",
    "Limit driver capabilities":
        "docker run --rm --gpus all -e NVIDIA_DRIVER_CAPABILITIES=compute,utility "
        "nvidia/cuda:12.4.1-base-ubuntu22.04 nvidia-smi",
    "NVENC video transcoding (needs 'video' cap)":
        "docker run --rm --gpus all -e NVIDIA_DRIVER_CAPABILITIES=compute,utility,video "
        "jrottenberg/ffmpeg:6-nvidia -hwaccel cuda -i in.mp4 -c:v h264_nvenc out.mp4",
    "Run a real PyTorch CUDA check":
        "docker run --rm --gpus all pytorch/pytorch:2.3.0-cuda12.1-cudnn8-runtime "
        "python -c 'import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))'",
}

for name, cmd in patterns.items():
    print(f"# {name}\n{cmd}\n")


## Advanced Features

### Container Device Interface (CDI)

CDI is the modern, runtime-agnostic way to expose GPUs. Instead of a special `nvidia` runtime shim, you generate a **CDI spec** describing each GPU as a named device, and the high-level runtime (Podman, containerd ≥ 1.7, CRI-O, Docker ≥ 25 with CDI enabled) applies the mounts directly.

```bash
# Generate a CDI spec for all GPUs (writes /etc/cdi/nvidia.yaml)
sudo nvidia-ctk cdi generate --output=/etc/cdi/nvidia.yaml

# List the device names the spec exposes (nvidia.com/gpu=all, =0, =GPU-<uuid>, =1:0 for MIG)
nvidia-ctk cdi list

# Run with Podman using a CDI device reference
podman run --rm --device nvidia.com/gpu=all nvidia/cuda:12.4.1-base-ubuntu22.04 nvidia-smi
```

Regenerate the spec whenever the driver is upgraded or GPUs/MIG layout change.

### Selecting GPUs and capabilities by environment

The toolkit reads these from the container environment (or `--gpus`):

- `NVIDIA_VISIBLE_DEVICES` — `all`, `none`, `void` (disable injection entirely), a comma list of indices, or GPU/MIG UUIDs.
- `NVIDIA_DRIVER_CAPABILITIES` — any of `compute`, `utility`, `graphics`, `video`, `display`, `compat32`, `ngx`, or `all`.
- `NVIDIA_REQUIRE_CUDA` — a constraint like `cuda>=12.0` that fails fast if the host driver is too old.

### containerd / nerdctl and Kubernetes

`nvidia-ctk runtime configure --runtime=containerd` adds the `nvidia` runtime handler to `/etc/containerd/config.toml`. In Kubernetes this handler is what the **NVIDIA device plugin** and **GPU Operator** target so that pods requesting `nvidia.com/gpu` resources land on a GPU-injecting runtime.

In [ ]:
# Inspect / set the toolkit runtime mode in /etc/nvidia-container-runtime/config.toml.
# Modes: "legacy" (prestart hook via the nvidia runtime shim), "cdi", or "csv" (Jetson).
EXAMPLE_CONFIG_TOML = r'''
# /etc/nvidia-container-runtime/config.toml (excerpt)
accept-nvidia-visible-devices-as-volume-mounts = false
accept-nvidia-visible-devices-envvar-when-unprivileged = true

[nvidia-container-cli]
  environment = []
  ldconfig = "@/sbin/ldconfig.real"

[nvidia-container-runtime]
  mode = "auto"          # auto | legacy | cdi | csv
  [nvidia-container-runtime.modes.cdi]
    default-kind = "nvidia.com/gpu"
'''
print(EXAMPLE_CONFIG_TOML)

# Switch a host to CDI mode non-interactively:
print("sudo nvidia-ctk config --set nvidia-container-runtime.mode=cdi --in-place")

# Verify what the toolkit will inject for a given device selection (dry run):
print("nvidia-container-cli --load-kmods info")          # list visible GPUs / driver version
print("nvidia-container-cli list --device 0")            # files/devices that would be mounted for GPU 0


## Use Cases

### Real-world Applications

#### Use Case 1: Multi-tenant training on a shared 8-GPU node

- **Context:** Several teams share one DGX-class node and must not see each other's GPUs.
- **Implementation:** Each job runs with `--gpus '"device=2,3"'` (or `NVIDIA_VISIBLE_DEVICES=GPU-<uuid>`); device cgroups created by `libnvidia-container` enforce isolation so a container literally cannot access GPUs it was not granted.
- **Result:** Hard per-container GPU partitioning without `--privileged`, schedulable by Slurm or Kubernetes.

#### Use Case 2: GPU-accelerated CI for a CUDA library

- **Context:** A team builds and unit-tests custom CUDA kernels on every PR.
- **Implementation:** CI runners are GPU hosts with the toolkit installed; the pipeline runs `docker run --gpus all <ci-image> ctest`. The image carries only the CUDA toolkit, so the same image works across runner generations as long as the host driver is new enough.
- **Result:** Reproducible GPU tests in ephemeral containers, no driver baked into images.

#### Use Case 3: Inference serving with Triton on Kubernetes

- **Context:** Serve TensorRT models at scale with autoscaling.
- **Implementation:** The NVIDIA device plugin (sitting on the toolkit's containerd runtime) advertises `nvidia.com/gpu`; pods request `resources.limits.nvidia.com/gpu: 1`, and the GPU Operator manages driver/toolkit/plugin lifecycle.
- **Result:** Declarative GPU scheduling; the toolkit handles the per-pod driver injection transparently.

## Best Practices

1. **Keep the host driver newer than every image's CUDA toolkit.** The host driver must satisfy each container's CUDA minor-version requirement. Use `NVIDIA_REQUIRE_CUDA=cuda>=X.Y` so mismatches fail fast instead of crashing deep in a job.
2. **Request only the capabilities you need.** `NVIDIA_DRIVER_CAPABILITIES=compute,utility` for training/inference; add `video` only for NVENC/NVDEC. Fewer injected libraries means a smaller attack surface and fewer surprises.
3. **Prefer CDI on new deployments.** It removes the special runtime shim, works uniformly across Podman/containerd/CRI-O, and is the direction NVIDIA is investing in. Regenerate the spec after every driver upgrade.
4. **Pin base images to explicit CUDA tags** (e.g. `nvidia/cuda:12.4.1-base-ubuntu22.04`), never `latest`. The `-base`, `-runtime`, and `-devel` variants matter: ship `-runtime` for serving, `-devel` only when you compile.
5. **Don't bake the driver into images.** If you find yourself copying `libcuda.so`, you've defeated the toolkit. Let the host provide it.
6. **Manage driver + toolkit + device plugin together in Kubernetes** with the GPU Operator rather than hand-installing each component.

## Common Pitfalls

1. **`could not select device driver "" with capabilities: [[gpu]]`** — the `nvidia` runtime isn't registered with Docker. Run `sudo nvidia-ctk runtime configure --runtime=docker && sudo systemctl restart docker`.
2. **Forgetting the host driver.** The toolkit injects but never installs the driver. If `nvidia-smi` fails on the *host*, no container will see a GPU — install the driver first.
3. **CUDA version too new for the host driver** — `CUDA driver version is insufficient for CUDA runtime version`. Upgrade the host driver or pin an older image. Driver ≥ toolkit, always.
4. **Missing the `video` capability** for transcoding — NVENC/NVDEC silently unavailable unless `NVIDIA_DRIVER_CAPABILITIES` includes `video`.
5. **Quoting `--gpus '"device=0,1"'` wrong** — the inner double quotes are required by the Docker CLI; without them you get "unknown device" errors. `NVIDIA_VISIBLE_DEVICES=0,1` avoids the quoting headache.
6. **Stale CDI spec after a driver upgrade** — device nodes/libraries change; regenerate `/etc/cdi/nvidia.yaml` or containers fail to start with missing-file errors.
7. **Expecting it to work in Docker Desktop on macOS/Windows.** GPU passthrough there is unsupported; use a native Linux host or WSL2 with the WSL CUDA driver.

## Performance Optimization

### Where the toolkit does and doesn't matter for performance

The toolkit only acts at **container start** (mounting libraries and devices); it adds **no runtime overhead** to GPU kernels once the container is up. So most "GPU container performance" work is really about the workload and host config, not the toolkit:

#### Configuration Tuning

- **Set IPC and shared memory for multi-worker data loaders.** Default `/dev/shm` (64 MB) starves PyTorch `DataLoader` workers — run with `--ipc=host` or `--shm-size=1g` to avoid bus errors and stalls.
- **Pin GPUs explicitly** with `NVIDIA_VISIBLE_DEVICES` rather than `all` on multi-GPU hosts so a job's CUDA context isn't created across unintended devices.
- **Enable NVLink/peer access for multi-GPU**: nothing special in the toolkit, but expose *all* the GPUs that NCCL will use together so peer-to-peer and NVLink topology are visible.
- **Use the `-runtime` image variant, not `-devel`,** for serving — smaller image, faster cold pulls and starts.

The cell below benchmarks container *startup* overhead (the only thing the toolkit influences) so you can see it is negligible relative to model load time.

In [ ]:
# Measure GPU-container startup overhead: minimal CUDA container vs. a no-op container.
# Run on a host with the toolkit installed. Uses subprocess so it works from a notebook.
import subprocess, time

def timed(cmd):
    start = time.perf_counter()
    subprocess.run(cmd, shell=True, check=False,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return time.perf_counter() - start

runs = 3
gpu_cmd = "docker run --rm --gpus all nvidia/cuda:12.4.1-base-ubuntu22.04 nvidia-smi -L"
cpu_cmd = "docker run --rm nvidia/cuda:12.4.1-base-ubuntu22.04 true"

try:
    gpu = sum(timed(gpu_cmd) for _ in range(runs)) / runs
    cpu = sum(timed(cpu_cmd) for _ in range(runs)) / runs
    print(f"avg no-op container start : {cpu*1000:7.1f} ms")
    print(f"avg GPU-injected start    : {gpu*1000:7.1f} ms")
    print(f"toolkit injection overhead: {(gpu-cpu)*1000:7.1f} ms  (one-time, at start)")
except Exception as exc:
    print("Requires Docker + the NVIDIA Container Toolkit on the host:", exc)


## Production Deployment

### Docker Compose

```yaml
# docker-compose.yml — request GPUs via the device reservation API
services:
  trainer:
    image: pytorch/pytorch:2.3.0-cuda12.1-cudnn8-runtime
    command: python train.py
    shm_size: "1gb"
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: all          # or `device_ids: ["0", "1"]`
              capabilities: [gpu]
```

### Kubernetes (device plugin / GPU Operator)

```yaml
# A pod requests whole GPUs as an extended resource. The NVIDIA device plugin
# (installed by the GPU Operator) advertises nvidia.com/gpu on each node, and
# the containerd "nvidia" runtime (configured by nvidia-ctk) injects the driver.
apiVersion: v1
kind: Pod
metadata:
  name: cuda-smoke
spec:
  restartPolicy: Never
  containers:
    - name: cuda
      image: nvidia/cuda:12.4.1-base-ubuntu22.04
      command: ["nvidia-smi"]
      resources:
        limits:
          nvidia.com/gpu: 1     # whole-GPU; MIG resources look like nvidia.com/mig-1g.10gb
```

Install the stack with the GPU Operator (it manages driver, toolkit, device plugin, DCGM, and node feature discovery):

```bash
helm repo add nvidia https://helm.ngc.nvidia.com/nvidia && helm repo update
helm install --wait gpu-operator nvidia/gpu-operator -n gpu-operator --create-namespace
```

## Monitoring and Observability

### Key Metrics to Track

- **GPU utilization & memory** (`DCGM_FI_DEV_GPU_UTIL`, `DCGM_FI_DEV_FB_USED/FREE`) — are containers actually using the GPUs they reserved?
- **Power & temperature** (`DCGM_FI_DEV_POWER_USAGE`, `DCGM_FI_DEV_GPU_TEMP`) — thermal throttling shows up here.
- **Per-process / per-container attribution** — DCGM-Exporter labels metrics with pod/container so you can bill or right-size.
- **XID errors** (`DCGM_FI_DEV_XID_ERRORS`) — driver/hardware faults that often surface as container crashes.

### Tooling

- **DCGM-Exporter** runs as a container/DaemonSet and exposes the metrics above to Prometheus; it's deployed automatically by the GPU Operator.
- `nvidia-smi` / `nvidia-smi dmon` from inside or outside the container for ad-hoc checks (requires the `utility` capability inside the container).
- `nvidia-container-cli --debug` and `/etc/nvidia-container-runtime/config.toml` `debug = "/var/log/nvidia-container-toolkit.log"` to trace injection itself.

### Logging Best Practices

- Enable toolkit debug logging only when diagnosing injection failures — it is verbose.
- Structure application logs to include the GPU UUID / index the job ran on, so failures can be correlated with a specific board.

## Troubleshooting

#### Issue 1: `docker: Error response from daemon: could not select device driver "" with capabilities: [[gpu]]`

**Symptoms:** `docker run --gpus all ...` fails immediately; `nvidia` runtime not found.

**Cause:** The Docker daemon doesn't have the NVIDIA runtime registered (toolkit installed but `daemon.json` not configured, or Docker not restarted).

**Solution:** `sudo nvidia-ctk runtime configure --runtime=docker && sudo systemctl restart docker`. Confirm with `docker info | grep -i runtime`.

#### Issue 2: `Failed to initialize NVML: Unknown Error` inside the container

**Symptoms:** Host `nvidia-smi` works, but it fails inside a running container — common after a `daemon-reload` or with `--privileged` + systemd cgroup churn.

**Cause:** A cgroup change reset the container's device access (a known interaction with `systemd` and cgroup v2 / `no-cgroups` settings).

**Solution:** Avoid `systemctl daemon-reload` against running GPU containers; set `no-cgroups = false` in the toolkit config, or restart the affected container. Pin GPUs by UUID to be safe.

#### Issue 3: `CUDA driver version is insufficient for CUDA runtime version`

**Symptoms:** Container starts but CUDA calls fail.

**Cause:** The container's CUDA toolkit is newer than the host driver supports.

**Solution:** Upgrade the host NVIDIA driver, or use an image with an older CUDA tag. Encode the requirement with `NVIDIA_REQUIRE_CUDA=cuda>=X.Y` to fail fast at start.

## Comparison with Alternatives

### How the NVIDIA Container Toolkit compares

| Aspect | NVIDIA Container Toolkit | `--privileged` + manual mounts | Driver baked into image |
|--------|-------------------------|--------------------------------|-------------------------|
| Driver source | Host (injected at runtime) | Host (hand-mounted) | Image (must match host exactly) |
| Image portability | High — one image, many hosts | Medium | Low — tied to one driver version |
| Security | Unprivileged, device-cgroup scoped | Broad — full host access | Unprivileged but brittle |
| Setup effort | `nvidia-ctk runtime configure` once | Error-prone per-run flags | Rebuild per driver bump |
| Orchestrator support | Docker, containerd, Podman, CRI-O, K8s | Ad-hoc | Ad-hoc |

**Legacy runtime (prestart hook) vs. CDI:** the classic mode uses the `nvidia-container-runtime` shim to inject a prestart hook; CDI moves the device description into a declarative spec the high-level runtime applies itself. CDI is runtime-agnostic and the recommended direction; the legacy mode remains the default for Docker for backward compatibility.

### When to Choose This Tool

- You run **any** CUDA/cuDNN/NCCL/TensorRT workload in containers — there is effectively no production alternative.
- You need portable images across hosts with differing driver versions.
- You want unprivileged, per-container GPU isolation and Kubernetes scheduling.

## Resources

### Official Documentation

- Install & user guide: https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/
- GitHub (toolkit): https://github.com/NVIDIA/nvidia-container-toolkit
- GitHub (libnvidia-container): https://github.com/NVIDIA/libnvidia-container
- CDI specification: https://github.com/cncf-tags/container-device-interface

### Related NVIDIA Cloud-Native Stack

- NVIDIA GPU Operator: https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/
- Kubernetes device plugin: https://github.com/NVIDIA/k8s-device-plugin
- DCGM / DCGM-Exporter: https://github.com/NVIDIA/dcgm-exporter
- NGC CUDA images: https://catalog.ngc.nvidia.com/orgs/nvidia/containers/cuda

### Tutorials and Guides

- Installing the toolkit (official): https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html
- Running a sample GPU workload with Docker: https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/sample-workload.html
- Architecture overview: https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/arch-overview.html

### Related Technologies

- Multi-Instance GPU (MIG) and time-slicing for GPU sharing
- containerd / nerdctl, Podman, CRI-O runtimes
- Triton Inference Server and TensorRT for serving